In [174]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score

df = pd.read_csv('customer_churn.csv', index_col=0)

# 1. Project Objective

Clearly state the purpose:

- Find what makes a customer leave?

What questions are we trying to answer?

- What makes a customer stay?

Why does this question matter to stakeholders?

- To keep the $ coming in.

# 2. Data Overview

In [175]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [176]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [177]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 
 17  

In [178]:
X = df.drop(columns=['Churn', 'PhoneService', 'TotalCharges', 'Contract'])

df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

y = df['Churn']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=.25, random_state=42)

In [179]:
numerical_cols = ['tenure', 'MonthlyCharges']
categorical_cols = df.drop(columns=['Contract', 'tenure', 'MonthlyCharges', 'PhoneService', 'TotalCharges', 'Churn']).columns.tolist()
#ordinal_cols = ["Contract"]

In [180]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,Churn
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,0.265370
std,0.368612,24.559481,30.090047,0.441561
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,0.000000
50%,0.000000,29.000000,70.350000,0.000000
75%,0.000000,55.000000,89.850000,1.000000
max,1.000000,72.000000,118.750000,1.000000


In [181]:
#This is where we left off.

In [182]:
# Creating pipeline



    
# }

preprocessor = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(sparse_output=False), categorical_cols),
    ('scaler', StandardScaler(), numerical_cols),
    #('Ordinal', OrdinalEncoder(), ordinal_cols)
])


pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100, 
        learning_rate=0.01, 
        random_state=42, 
        eval_metric="logloss", 
        verbosity=0))
])


pipe.fit(X_train, y_train)

#y_pred = pipe.predict(X_test)
#acc_score = accuracy_score(y_test, y_pred)

y_train_pred = pipe.predict(X_train)
acc_score_train = accuracy_score(y_train, y_train_pred)

print(acc_score_train)
# rs = RandomizedSearchCV(pipe, param_grid, n_iter=20, random_state=42)
# rs.fit(X_train, y_train)

0.8144378698224852


In [184]:
param_grid = {
    "model__n_estimators": [100, 200, 300, 400, 500, 1000],
    "model__learning_rate": [.1, .08, .06, .04, .02, .01]
}

gs = GridSearchCV(pipe, param_grid, cv=5, scoring="precision", n_jobs=-1)

gs.fit(X_temp, y_temp)

,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'model__learning_rate': [0.1, 0.08, ...], 'model__n_estimators': [100, 200, ...]}"
,scoring,'precision'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('ohe', ...), ('scaler', ...)]"


In [185]:
for k, v in gs.best_params_.items():
    print(f"  {k}: {v}")
print("Best CV R2 :", round(gs.best_score_, 3))   # avg across 5 folds
print("Test f1    :", round(gs.score(X_test, y_test), 3))  # final hold-out

  model__learning_rate: 0.01
  model__n_estimators: 100
Best CV R2 : 0.768
Test f1    : 0.818


In [186]:
# Validation dataset

y_pred_val = pipe.predict(X_val)
acc_score_val = accuracy_score(y_val, y_pred_val)
precision = precision_score(y_val, y_pred_val)
recall = recall_score(y_val, y_pred_val)

print("accuracy: ", acc_score_val)
print("precision: ", precision)
print("recall: ", recall)


accuracy:  0.7835344215755855
precision:  0.6887755102040817
recall:  0.3562005277044855


In [187]:
# Test dataset

y_pred_test = pipe.predict(X_test)
acc_score_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)

print("accuracy: ", acc_score_test)
print("precision: ", precision_test)
print("recall: ", recall_test)

accuracy:  0.7984386089425124
precision:  0.7405405405405405
recall:  0.3672922252010724


# 3. Data Overview

1. Which features showed the strongest relationship/predictive potential?
-  Fiber optics/internet service is #1, and locking your customers into a contract prevents them from leaving.

In [188]:
importances = pd.Series(pipe.named_steps['model'].feature_importances_, index=pipe.named_steps['preprocessor'].get_feature_names_out()).sort_values(ascending=False)

print(importances)

ohe__TechSupport_No                             0.273249
ohe__OnlineSecurity_No                          0.270788
ohe__InternetService_Fiber optic                0.075446
ohe__InternetService_DSL                        0.072120
scaler__tenure                                  0.051889
ohe__PaymentMethod_Electronic check             0.033129
ohe__MultipleLines_No                           0.021082
ohe__OnlineBackup_No                            0.016733
ohe__StreamingMovies_No                         0.016555
ohe__PaperlessBilling_No                        0.016205
scaler__MonthlyCharges                          0.014017
ohe__MultipleLines_No phone service             0.013874
ohe__StreamingTV_No                             0.013776
ohe__StreamingMovies_Yes                        0.013370
ohe__PaymentMethod_Mailed check                 0.012937
ohe__SeniorCitizen_0                            0.012314
ohe__OnlineBackup_Yes                           0.010838
ohe__DeviceProtection_No       

In [196]:

lin_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lin_pipe.fit(X_temp, y_temp)

#print(lin_pipe.named_steps["model"].coef_)


lin_score= {}

for name, coef in zip(lin_pipe.named_steps['preprocessor'].get_feature_names_out(), lin_pipe.named_steps["model"].coef_):
    lin_score[name] = coef
    #print(f"  {name}: {coef:.2f}")

lin_score_sorted = pd.Series(lin_score).sort_values(ascending=False)
lin_score_sorted

ohe__InternetService_Fiber optic                0.159530
ohe__StreamingMovies_Yes                        0.068143
ohe__PaymentMethod_Electronic check             0.065036
ohe__MultipleLines_Yes                          0.060637
ohe__StreamingTV_Yes                            0.058739
ohe__OnlineSecurity_No                          0.047755
ohe__TechSupport_No                             0.045692
ohe__OnlineBackup_No                            0.029282
ohe__PaperlessBilling_Yes                       0.025254
ohe__DeviceProtection_Yes                       0.023131
ohe__SeniorCitizen_1                            0.021563
ohe__DeviceProtection_No                        0.019732
ohe__OnlineBackup_Yes                           0.013580
ohe__Dependents_No                              0.012525
ohe__gender_Female                              0.004938
ohe__Partner_Yes                                0.002507
ohe__Partner_No                                -0.002507
ohe__TechSupport_Yes           

In [189]:
# Churn percentage
churn_perc = (round(df.groupby("Contract")["Churn"].mean() * 100, 0)).astype(int)
churn_perc

Contract
Month-to-month    43
One year          11
Two year           3
Name: Churn, dtype: int64

# 4. Modeling Approach

Model used was XGB classifier. 
parameters were determined by utilizing gridsearchCV to look at the best precision score. Validation data set made from splitting was used to tweak and verify.

1. Contract, internet service, 

2. Model peformance:

on validation set:
-  accuracy:  0.7814052519517388
-  precision:  0.7028571428571428
-  recall:  0.3245382585751979

on test set:
-  accuracy:  0.8019872249822569
-  precision:  0.7554347826086957
-  recall:  0.3726541554959786


Model: XGBClassifier
-  n_estimators=100
-  learning_rate=0.01 
-  random_state=42

# 5. Key Results and Recommendations

-  Fix your fiber network and put in the contract that no other compaines can utilize your infrastructure
-  Lock all customers into 1-2 year contracts